# Treinamento de IA

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle #Salvar tradutores

#Para usar o fasttext, tive que importar a biblioteca do gensim
import gensim
from gensim.models import KeyedVectors

# TensorFlow e Keras (tive que seperar para evitar erro)
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional # type: ignore
from tensorflow.keras.models import load_model # type: ignore
from tensorflow.keras.callbacks import EarlyStopping # type: ignore

# Biblioteca do sklearn
from sklearn.preprocessing import LabelEncoder

#Path ou caminhos
path_models = '../models/'
path_dados = '../data/iniciacao.csv' #preciso de um dataset centralizado com todas as informações
path_fasttext = '../data/cc.pt.300.vec.gz'

#Caso alguém exclua a pasta, e estou fazendo muito isso inclusive
if not os.path.exists(path_models):
    os.makedirs(path_models)

# Parametros (tive que criar para mexer manualmente, sem ler o codigo completo)
vocab = 20000 #limite do vocabulario (por segurança)
maxlen = 20 #tamanho maximo das frases
epochs = 100 #as epocas
batch_size = 32
unidades_lstm = 128
embedding_dim = 300 #isso são as dimensõoes para o fasttext

#teste com o fasttext
acertos = 0
falhas = 0

# Vetorização 
df = pd.read_csv(path_dados) #vetorizar os dados dentro do csv
df = df.drop_duplicates() #remover duplicatas para o treino (ainda vou continuar reiniciando o kernel por garantia)
word_fasttext = KeyedVectors.load_word2vec_format(path_fasttext, limit=100000)

#pegar os valores direto
perguntas = df['perguntas'].values
respostas = df['respostas'].values

#print de teste
print(f"Encontradas {len(perguntas)} perguntas e {len(respostas)} respostas.")

#todo o processo de desenvolvimento da RNN será uma LSTM com o Keras API
#Sequências (x)
tokenizer = Tokenizer(num_words=vocab, oov_token="<OOV>") #limite de palavras e o oov_token para palavras fora do vocabulario
tokenizer.fit_on_texts(perguntas)
vocab_prop = len(tokenizer.word_index) + 1 #proporção do vocabulario
x_seq = tokenizer.texts_to_sequences(perguntas) #transforma todas as palavras em sequencias numericas (coluna x)
x_processadas = pad_sequences(x_seq, maxlen=maxlen, padding='post')

#prints de teste
#print("Exemplo de X (bruto):", perguntas[0])
#print("Exemplo de X (processado):", x_processadas[0])

#Processo de Codificação Categórica (Y)
label_encoder = LabelEncoder()
y_respostas = label_encoder.fit_transform(np.array(respostas)) #transforma as respostas em numeros, aplicado o np.array para evitar erro
y_respostas = np.array(y_respostas)  # Força conversão para NumPy array, apenas para o vscode não reclamar
#aparentemente o LabelEncoder não aceita Series do pandas diretamente

#prints de teste
#print("Exemplo de Y (bruto):", respostas[0])
#print("Exemplo de Y (Label):", y_respostas[0])

#vou converter os IDs para One-Hot Encoding, trabalhar apenas com o metodo binario, isso facilita a vida da rede neural, treinamento e categorização
num_classes = len(np.unique(y_respostas)) #calcular o output unico
y_categorico = to_categorical(y_respostas, num_classes=num_classes)

print(f"Total de classes (respostas únicas): {num_classes}")
print("Exemplo de Y (Categorical/One-Hot):", y_categorico[0])

with open(path_models + 'tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open(path_models + 'label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)


#Carregar o FastText
embedding_matrix = np.zeros((vocab_prop, embedding_dim)) #criar um matrix com o fasttext

#Verificar o vocabulário da IA com o FastTest
for word, i in tokenizer.word_index.items():
    if i >= vocab_prop: continue
    if word in word_fasttext:
        embedding_matrix[i] = word_fasttext[word]
        acertos += 1
    else:
        falhas += 1

print(f"Matriz pronta! {acertos} palavras encontradas. {falhas} não encontradas (gírias/erros).")


model = Sequential()

#Embedding transforma os IDs em vetores densos e é aqui que ela ira aprender o "significado" e contexto das palavras
#aplicado o fastext (um cérebro adulto)
model.add(Embedding(input_dim=vocab_prop, 
                    output_dim=300,
                    weights=[embedding_matrix], #injetando o fasttext
                    trainable=False, #congelar aprendizado
                    input_length=maxlen))

'''
#Primeira camada LSTM
model.add(LSTM(units=unidades_lstm, return_sequences=True))
model.add(Dropout(0.3))

#Segunda camada LSTM
model.add(LSTM(units=unidades_lstm // 2)) #metade das unidades
model.add(Dropout(0.3))

#tentei efetuar o processo em duas camadas, mas o empilhamento bidirecional é mais eficiente
'''
#tentando um moeto de empilhamento bidirecional
model.add(Bidirectional(LSTM(units=unidades_lstm)))
model.add(Dropout(0.3))

#É a saida de decisão, pega os valores de LSTM e decide a resposta a retornar
model.add(Dense(units=num_classes, activation='softmax'))

#Treinar o Modelo
model.compile(loss='categorical_crossentropy', # Função de perda para classificação
              optimizer='adam',                 # Otimizador padrão
              metrics=['accuracy'])             # Queremos ver a acurácia
model.summary() #mostrar arquitetura

'''
fazer o uso do val_loss dentro do monitoramento é mais preciso para a aprendizagem da IA
val_accuracy ou a propia accuracy não é ensinar, apenas decorar as palavras com respostas 
'''
early_stopping = EarlyStopping(monitor='val_loss',
                               patience=15, 
                               restore_best_weights=True)

history = model.fit(x_processadas, 
                    y_categorico, 
                    epochs=epochs, 
                    batch_size=batch_size, 
                    validation_split=0.2,
                    callbacks=[early_stopping])

#Salvar o modelo que treinei
model.save('../models/chatbotIA.keras') #antes estava usando o h5, mas era uma forma legada, atualizado para o proprio keras


Encontradas 75 perguntas e 75 respostas.
Total de classes (respostas únicas): 7
Exemplo de Y (Categorical/One-Hot): [0. 0. 0. 0. 1. 0. 0.]
Matriz pronta! 91 palavras encontradas. 1 não encontradas (gírias/erros).


c:\Users\playe\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_14 (Embedding)        │ ?                      │        27,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_13                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,900 (108.98 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 27,900 (108.98 KB)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 318ms/step - accuracy: 0.1167 - loss: 1.9530 - val_accuracy: 0.0000e+00 - val_loss: 1.9619
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.3833 - loss: 1.9047 - val_accuracy: 0.0000e+00 - val_loss: 2.0000
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.5333 - loss: 1.8594 - val_accuracy: 0.0000e+00 - val_loss: 2.0446
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5500 - loss: 1.8106 - val_accuracy: 0.0000e+00 - val_loss: 2.0993
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.5167 - loss: 1.7583 - val_accuracy: 0.0000e+00 - val_loss: 2.1718
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.4833 - loss: 1.6740 - val_accuracy: 0.0000e+00 - val_loss: 2.2822
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.4500 - loss: 1.5886 - val_accuracy: 0.0000e+00 - val_loss: 2.4693
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.4000 - loss: 1.4859 - val

# Graficos

Anteriormente estava fazendo diversos show(), mas é inviavel analisar dessa forma e estou gerando png com os graficos

In [31]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import confusion_matrix

#Configurar Past
path_reports = '../reports/'

#Só um receio caso seja deletado, visto que vou ter que retirar o .gitkeep da pasta
if not os.path.exists(path_reports):
    os.makedirs(path_reports)
    print(f"Pasta '{path_reports}' foi criada com sucesso")


#Extração do histórico de treinamento
acc = history.history['accuracy'] #acurácia de treino
val_acc = history.history['val_accuracy'] #acurácia de validação
loss = history.history['loss'] #perda de treino
val_loss = history.history['val_loss'] #perda de validação
epochs = range(1, len(acc) + 1) #número de épocas treinadas


# Subplot de Acurácia e Perca
plt.figure(figsize=(16, 6)) #tamanho do grafico

#1. Grafico: Acurácia
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Acurácia (Treino)')
plt.plot(epochs, val_acc, label='Acurácia (Validação)')
plt.legend(loc='upper left')
plt.title('Acurácia de Treinamento vs. Validação')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.grid(True, alpha=0.3)

#2. Grafico: Perca 
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Perda (Treino)')
plt.plot(epochs, val_loss, label='Perda (Validação)')
plt.legend(loc='upper left')
plt.title('Perda de Treinamento vs. Validação')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.grid(True, alpha=0.3)

#para salvar os dois graficos
arquivo = path_reports + 'grafico_acuracia_perca.png'
plt.savefig(arquivo, bbox_inches='tight') #salvar o arquivo
plt.close() #fechameto para dar continuidade

#3. Grafico: Balanceamento de Classes
plt.figure(figsize=(10, 8))
sns.countplot(y=respostas)
plt.title('Distribuição de Classes nas Respostas')
plt.xlabel('Contagem')
plt.ylabel('Respostas')

arquivo = path_reports + 'balanceamento_de_classes.png'
plt.savefig(arquivo, bbox_inches='tight') 
plt.close() 


#4. Grafico: Matriz de Confusão
#Gerar previsões
y_pred_prob = model.predict(x_processadas)
y_pred_classes = np.argmax(y_pred_prob, axis=1) # Pega a classe com maior probabilidade
y_true_classes = y_respostas  # 'y_respostas' são os resultados (são os IDs)

#Gerar a matriz e mapear
cm = confusion_matrix(y_true_classes, y_pred_classes)
class_names = label_encoder.classes_.astype(str).tolist()

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusão')
plt.xlabel('Previsto pelo Modelo')
plt.ylabel('Valor Real')

arquivo = path_reports + 'matriz_confusao.png'
plt.savefig(arquivo, bbox_inches='tight') 
plt.close() 


#Distribuição do Comprimento das Frases
# Calcular o número de palavras em cada pergunta
df['comprimento_pergunta'] = df['perguntas'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(data= df, x='comprimento_pergunta', bins=15, kde=True)
plt.title('Distribuição do Número de Palavras por Pergunta')
plt.xlabel('Número de Palavras')
plt.ylabel('Frequência')
# Linha vertical para mostrar nosso 'maxlen'
plt.axvline(x=20, color='red', linestyle='--', label=f'maxlen = {20}')
plt.legend()

arquivo = path_reports + 'distribuicao_frases.png'
plt.savefig(arquivo, bbox_inches='tight') 
plt.close() 

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
